# KDD: Limpieza, resolución de entidades y hechos

Se limpia el staging, se resuelven proveedores y entidades duplicados y se construye la tabla de hechos con grano
**entidad–proveedor–ítem** en `data/final/`, particionada por `anio` y `mes`

### Imports

In [1]:
import os, socket, time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import ArrayType, StructType

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.max_colwidth", 80)

### Utilidades

- Constantes

In [ ]:
NOTEBOOK = "a_transformaciones"

proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent

raw_dir     = proyecto / "data" / "raw"
bronze_dir  = proyecto / "data" / "bronze"
staging_dir = proyecto / "data" / "staging"
final_dir   = proyecto / "data" / "final"
artef_dir   = proyecto / "data" / "artefactos"
tc_path     = proyecto / "data" / "cambios.csv"
anios = [2023, 2024, 2025]

out_dir = proyecto / "01_EDA_KDD" / "temp" / NOTEBOOK
fig_dir, tab_dir, web_dir = out_dir / "figs", out_dir / "tablas", out_dir / "web"
for d in (fig_dir, tab_dir, web_dir):
    d.mkdir(parents=True, exist_ok=True)

def mostrar_fig(nombre):
    plt.tight_layout()
    plt.savefig(fig_dir / f"{nombre}.png", dpi=150, bbox_inches="tight")
    plt.show()

def guardar_tabla(df, nombre):
    sdf = spark.createDataFrame(df) if isinstance(df, pd.DataFrame) else df
    sdf.coalesce(1).write.mode("overwrite").parquet(str(tab_dir / nombre))
    sdf.toPandas().to_json(web_dir / f"{nombre}.json", orient="records", force_ascii=False, date_format="iso", indent=1)
    return len(df) if isinstance(df, pd.DataFrame) else sdf.count()

def leer(directorio, nombre):
    return spark.read.parquet(str(directorio / nombre))

- Spark

In [3]:
spark_master = os.environ.get("spark_master", "local[*]")

builder = (SparkSession.builder
           .appName(NOTEBOOK)
           .master(spark_master)
           .config("spark.driver.memory",   os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
           .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "4g"))
           .config("spark.sql.shuffle.partitions", "24")
           .config("spark.ui.showConsoleProgress", "false")
           .config("spark.sql.session.timeZone", "America/Lima"))

if spark_master.startswith("spark://"):
    builder = (builder.config("spark.driver.host", socket.gethostname())
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
SHUFFLE = int(spark.conf.get("spark.sql.shuffle.partitions"))

### Verificación del cluster

In [4]:
sc = spark.sparkContext
estado = sc._jsc.sc().getExecutorMemoryStatus()

print(f"Master        : {sc.master}")
print(f"Executors     : {estado.size() - 1}")
print(f"Cores totales : {sc.defaultParallelism}")
print(f"Memoria driver: {sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3:.1f} GB")
print("UI del job    : http://localhost:4040")

Master        : spark://spark-master:7077
Executors     : 0
Cores totales : 2
Memoria driver: 2.0 GB
UI del job    : http://localhost:4040


### Carga del staging

In [5]:
proc_raw   = leer(staging_dir, "procesos")
adj_raw    = leer(staging_dir, "adjudicaciones").drop("anio")
of_raw     = leer(staging_dir, "oferentes").drop("anio")
items_raw  = leer(staging_dir, "items").drop("anio")
con_raw    = leer(staging_dir, "contratos").drop("anio")
partes_raw = leer(staging_dir, "partes").where(F.col("anio").isin(anios)).drop("anio")

---
### 1. Limpieza

#### a. Auxiliares (los del P1, con espacios colapsados y moneda/fecha parametrizables)

In [6]:
def normalizar_texto(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.regexp_replace(F.upper(F.trim(F.col(c))), r"\s+", " "))
    return df

def quitar_duplicados(df):
    w = Window.partitionBy("ocid").orderBy(F.col("fecha").desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

def filtrar_anios(df, anios):
    return df.where(F.col("anio").isin(anios))

SERIES_TC = {"USD": "PN01234PM", "EUR": "PN01235PM", "BRL": "PN01237PM", "GBP": "PN01238PM", "CAD": "PN01244PM"}
MESES = {"Ene": 1, "Feb": 2, "Mar": 3, "Abr": 4, "May": 5, "Jun": 6, "Jul": 7, "Ago": 8, "Sep": 9, "Set": 9,
         "Oct": 10, "Nov": 11, "Dic": 12}

def descargar_tipo_cambio(ruta, anios):
    import requests
    filas = []
    for moneda, serie in SERIES_TC.items():
        url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{serie}/json/{min(anios)}-1/{max(anios)}-12"
        for periodo in requests.get(url, timeout=30).json()["periods"]:
            mes, anio = periodo["name"].split(".")
            filas.append((moneda, int(anio), MESES[mes], float(periodo["values"][0])))
    pd.DataFrame(filas, columns=["moneda", "anio", "_mes", "tc"]).to_csv(ruta, index=False)

def convertir_a_soles(df, tc, col_monto, col_moneda, col_fecha):
    tc_ = tc.withColumnRenamed("moneda", "_mon").withColumnRenamed("anio", "_anio")
    return (df.withColumn("_mon", F.coalesce(F.col(col_moneda), F.lit("PEN")))
              .withColumn("_anio", F.year(col_fecha)).withColumn("_mes", F.month(col_fecha))
              .join(F.broadcast(tc_), ["_mon", "_anio", "_mes"], "left")
              .withColumn(col_monto, F.when(F.col("_mon") == "PEN", F.col(col_monto)).otherwise(F.col(col_monto) * F.col("tc")))
              .drop("_mon", "_anio", "_mes", "tc"))

def limpiar_montos(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.when(F.col(c) > 0, F.col(c)))
    return df

#### b. Flujo

In [7]:
if not tc_path.exists():
    descargar_tipo_cambio(tc_path, anios)
tipo_cambio = spark.createDataFrame(pd.read_csv(tc_path))

pasos = [("procesos staging", proc_raw.count())]
proc = normalizar_texto(proc_raw, ["entidad", "region", "metodo_detalle", "categoria"])
proc = quitar_duplicados(proc);    pasos.append(("sin duplicados", proc.count()))
proc = filtrar_anios(proc, anios); pasos.append((f"fecha en {min(anios)}-{max(anios)}", proc.count()))
proc = convertir_a_soles(proc, tipo_cambio, "monto_referencial", "moneda", "fecha")
proc = limpiar_montos(proc, ["monto_referencial"])
proc = (proc.withColumn("mes", F.month("fecha"))
            .withColumn("postor_unico", (F.col("n_postores") == 1).cast("int"))
            .withColumn("es_directa", F.col("metodo_detalle").rlike("CONTRATACI.N DIRECTA").cast("int")))

ESTADOS_EXCLUIDOS = ["cancelled", "unsuccessful"]
adj = (adj_raw.join(proc.select("ocid", F.col("fecha").alias("_fp")), "ocid")      # solo procesos válidos
              .withColumn("_f", F.coalesce("fecha_adj", "_fp"))
              .withColumn("award_id", F.coalesce("award_id", F.lit("SIN_ID"))))
pasos.append(("filas adj en procesos válidos", adj.count()))
adj = limpiar_montos(convertir_a_soles(adj, tipo_cambio, "monto_adj", "moneda_adj", "_f"), ["monto_adj"]).drop("_f", "_fp")
adj = adj.where(~F.lower(F.coalesce("estado_adj", F.lit(""))).isin(ESTADOS_EXCLUIDOS)).cache()
pasos.append(("adj sin cancelled/unsuccessful", adj.count()))
pasos_df = pd.DataFrame(pasos, columns=["paso", "filas"])
pasos_df

,paso,filas
0,procesos staging,236254
1,sin duplicados,236254
2,fecha en 2023-2025,235977
3,filas adj en procesos válidos,203329
4,adj sin cancelled/unsuccessful,203329


---
### 2. Resolución de entidades

En tres niveles, del más seguro al menos seguro:

1. **Identificador** (determinista): se extrae el RUC (11 dígitos) aunque venga con prefijo o sufijo; un RUC 10 se
   convierte en el DNI de la persona natural (`10 + DNI + dígito` es la misma persona) → clave `PN:`
2. **Nombre normalizado** (determinista): mayúsculas, sin tildes, sin puntuación y sin forma societaria
   (`S.A.C.` = `SOCIEDAD ANONIMA CERRADA` = `SAC`) → nombre canónico por clave
3. **MinHash + LSH por bandas** (probabilístico, como en el P1): solo para menciones **sin** identificador; se enlazan a la
   clave más parecida si Jaccard ≥ 0.85, si no se crea un nodo `NOM:`

No se fusionan claves distintas aunque el nombre sea casi idéntico: dos RUC son dos personas jurídicas. Esos pares se
guardan como "posibles vinculadas", como señal para revisión, no como hecho

In [8]:
CON_TILDE, SIN_TILDE = "ÁÉÍÓÚÜÑ", "AEIOUUN"
FORMAS = (r"\b(SOCIEDAD ANONIMA CERRADA|SOCIEDAD ANONIMA ABIERTA|SOCIEDAD ANONIMA|"
          r"SOCIEDAD COMERCIAL DE RESPONSABILIDAD LIMITADA|EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA|"
          r"S A C|S A A|S R L|E I R L|SAC|SAA|SRL|EIRL|LTDA|S A)\b")

def normalizar_nombre(c):
    x = F.translate(F.upper(F.trim(F.col(c))), CON_TILDE, SIN_TILDE)
    x = F.regexp_replace(x, r"[^A-Z0-9 ]", " ")
    x = F.regexp_replace(F.regexp_replace(x, r"\s+", " "), FORMAS, " ")
    return F.coalesce(F.trim(F.regexp_replace(x, r"\s+", " ")), F.lit(""))

def clave_id(c):
    s = F.coalesce(F.col(c).cast("string"), F.lit(""))
    d11 = F.regexp_extract(s, r"(\d{11})", 1)
    d8 = F.regexp_extract(s, r"(?<!\d)(\d{8})(?!\d)", 1)
    return (F.when(d11.startswith("10"), F.concat(F.lit("PN:"), F.substring(d11, 3, 8)))
             .when(d11 != "", F.concat(F.lit("RUC:"), d11))
             .when(d8 != "", F.concat(F.lit("PN:"), d8)))

#### a. Entidades compradoras

La clave es el RUC; los homónimos (p. ej. municipalidades distritales con el mismo nombre) **no** se fusionan

In [9]:
proc = proc.withColumn("entidad_key", F.coalesce(clave_id("entidad_id_raw"),
                                                 F.concat(F.lit("NOM:"), F.substring(F.sha1("entidad"), 1, 12)))).cache()
pasos_df.loc[len(pasos_df)] = ("procesos limpios", proc.count())

w = Window.partitionBy("entidad_key").orderBy(F.desc("n"), "entidad")
dim_entidades = (proc.groupBy("entidad_key", "entidad", "region").agg(F.count("*").alias("n"))
                     .withColumn("rk", F.row_number().over(w)).where("rk = 1")
                     .select("entidad_key", F.col("entidad").alias("nombre"), "region"))
homonimos = proc.groupBy("entidad").agg(F.countDistinct("entidad_key").alias("claves")).where("claves > 1")
print(f"entidades: {dim_entidades.count():,} claves | {proc.select('entidad').distinct().count():,} nombres | "
      f"homónimos legítimos: {homonimos.count():,}")

entidades: 2,995 claves | 2,995 nombres | homónimos legítimos: 0


#### b. Catálogo de menciones de proveedores (adjudicados + oferentes)

In [10]:
menciones = (adj.select(F.lit("proveedor").alias("rol"), "ocid", F.col("proveedor_id_raw").alias("id_raw"),
                        F.col("proveedor").alias("nombre_raw"))
    .unionByName(of_raw.join(proc.select("ocid"), "ocid")
                 .select(F.lit("oferente").alias("rol"), "ocid", F.col("oferente_id_raw").alias("id_raw"),
                         F.col("oferente").alias("nombre_raw")))
    .withColumn("clave", clave_id("id_raw"))
    .withColumn("nombre_norm", normalizar_nombre("nombre_raw"))
    .where("clave IS NOT NULL OR nombre_norm <> ''")
    .cache())

w = Window.partitionBy("clave").orderBy(F.desc("n"), "nombre")
canon = (menciones.where("clave IS NOT NULL")
         .groupBy("clave", F.upper(F.trim("nombre_raw")).alias("nombre"), "nombre_norm").agg(F.count("*").alias("n"))
         .withColumn("rk", F.row_number().over(w)).cache())
nombres_clave = canon.where("rk = 1").select("clave", "nombre")
variantes_clave = canon.where("nombre_norm <> ''").select("clave", "nombre_norm").distinct()
multi_variante = (canon.groupBy("clave").agg(F.countDistinct("nombre_norm").alias("variantes"),
                                             F.collect_set("nombre").alias("ejemplos")).where("variantes > 1"))
sin_clave = menciones.where("clave IS NULL").groupBy("nombre_norm").agg(F.count("*").alias("menciones")).cache()
multi_variante.orderBy(F.desc("variantes")).show(5, truncate=90)

+---------------+---------+------------------------------------------------------------------------------------------+
|          clave|variantes|                                                                                  ejemplos|
+---------------+---------+------------------------------------------------------------------------------------------+
|RUC:20571170958|        4|[CORPORACION INTERNACIONAL GRUPO TOPCONSA S.A.C. - GRUPO TOPCONSA S.A.C., CONSTRUCTORA ...|
|RUC:20529104988|        3|[INVERSIONES EN TI JESUS SOCIEDAD ANONIMA CERRADA, COMERCIALIZACIONES REPRESENTACIONES ...|
|RUC:20552504641|        3|                       [OPTICAL TECHNOLOGIES SAC, ON EMPRESAS S.A.C., WIN EMPRESAS S.A.C.]|
|RUC:20600212070|        3|    [AGS CONSTRUCTORES GENERALES S.A.C., AGS INVERSIONES S.A.C., ARGOS CORPORATION S.A.C.]|
|RUC:20601598567|        3|[GRUPO DICOS S.R.L., CORPORACION NUTRIANDINA SOCIEDAD COMERCIA DE RESPONSABILIDAD LIMIT...|
+---------------+---------+---------------------

#### c. MinHash + LSH por bandas (AND-OR) con funciones de Spark

In [11]:
K, B = 30, 6
R = K // B                    # umbral ≈ (1/6)^(1/5) ≈ 0.70 · P(cand|0.85) ≈ 0.97 · P(cand|0.5) ≈ 0.17
MAX_BUCKET, UMBRAL_LINK, UMBRAL_VINC = 100, 0.85, 0.95
CALCULAR_VINCULADAS = True    # auditoría; ponlo en False si aún demora
tmp_dir = proyecto / "data" / "tmp_er"

def con_shingles(df, k=3):
    return df.withColumn("sh", F.when(F.length("nombre_norm") >= k, F.array_distinct(
        F.expr(f"transform(sequence(1, length(nombre_norm) - {k} + 1), i -> substring(nombre_norm, i, {k}))")))
        .otherwise(F.array("nombre_norm")))

def minhash(i):
    return F.array_min(F.transform("sh", lambda s: F.xxhash64(s, F.lit(i))))

def jaccard(a, b):
    return F.size(F.array_intersect(a, b)) / F.size(F.array_union(a, b))

def materializar(df, nombre):
    df.write.mode("overwrite").parquet(str(tmp_dir / nombre))
    return spark.read.parquet(str(tmp_dir / nombre))

t0 = time.time()
nombres_con = variantes_clave.select("nombre_norm").distinct()

# 1) nombre sin id idéntico a un nombre con id (y una sola clave) → enlace directo, sin LSH
exactos = (sin_clave.select("nombre_norm").join(variantes_clave, "nombre_norm")
           .groupBy("nombre_norm").agg(F.min("clave").alias("clave"), F.countDistinct("clave").alias("_n"))
           .where("_n = 1").drop("_n")
           .withColumn("nombre_b", F.col("nombre_norm")).withColumn("jaccard", F.lit(1.0)))
pendientes = sin_clave.select("nombre_norm").join(nombres_con, "nombre_norm", "left_anti")

# 2) shingles y bandas UNA vez por nombre distinto, materializados
sh = materializar(con_shingles(pendientes.unionByName(nombres_con).distinct()), "sh")
llaves = lambda: [F.xxhash64(*[F.col("firma")[b * R + j] for j in range(R)]) for b in range(B)]
bandas = materializar(sh.withColumn("firma", F.array(*[minhash(i) for i in range(K)]))
                        .select("nombre_norm", F.posexplode(F.array(*llaves())).alias("banda", "bucket")), "bandas")

b_con = bandas.join(nombres_con, "nombre_norm")
calientes = b_con.groupBy("banda", "bucket").count().where(f"count > {MAX_BUCKET}")
b_con = materializar(b_con.join(calientes.select("banda", "bucket"), ["banda", "bucket"], "left_anti"), "b_con")
b_pen = bandas.join(pendientes, "nombre_norm")
print(f"pendientes: {pendientes.count():,} | nombres con id: {nombres_con.count():,} | buckets calientes: {calientes.count():,}")

def con_jaccard(pares):
    sa = sh.select(F.col("nombre_norm").alias("nombre_a"), F.col("sh").alias("sh_a"))
    sb = sh.select(F.col("nombre_norm").alias("nombre_b"), F.col("sh").alias("sh_b"))
    return pares.join(sa, "nombre_a").join(sb, "nombre_b").withColumn("jaccard", jaccard("sh_a", "sh_b")).drop("sh_a", "sh_b")

# 3) candidatos solo con nombres (sin arrays en el shuffle); Jaccard exacto después
pares = (b_pen.alias("a").join(b_con.alias("b"), ["banda", "bucket"])
         .select(F.col("a.nombre_norm").alias("nombre_a"), F.col("b.nombre_norm").alias("nombre_b")).distinct())
cand = (con_jaccard(pares).where(f"jaccard >= {UMBRAL_LINK}")
        .join(variantes_clave.withColumnRenamed("nombre_norm", "nombre_b"), "nombre_b"))
w = Window.partitionBy("nombre_a").orderBy(F.desc("jaccard"), "clave")
enlaces = materializar(
    cand.withColumn("rk", F.row_number().over(w)).where("rk = 1")
        .select(F.col("nombre_a").alias("nombre_norm"), "clave", "nombre_b", "jaccard")
        .unionByName(exactos), "enlaces")

mapa_sin = materializar(
    sin_clave.join(enlaces, "nombre_norm", "left")
        .withColumn("metodo", F.when(F.col("clave").isNull(), "nuevo")
                               .when(F.col("jaccard") == 1.0, "exacto").otherwise("lsh"))
        .withColumn("clave", F.coalesce("clave", F.concat(F.lit("NOM:"), F.substring(F.sha1("nombre_norm"), 1, 12)))),
    "mapa_sin")

# 4) auditoría: RUC distintos con nombre casi idéntico (no se fusionan)
if CALCULAR_VINCULADAS:
    pares_v = (b_con.alias("x").join(b_con.alias("y"), ["banda", "bucket"])
               .where(F.col("x.nombre_norm") < F.col("y.nombre_norm"))
               .select(F.col("x.nombre_norm").alias("nombre_a"), F.col("y.nombre_norm").alias("nombre_b")).distinct())
    vc = variantes_clave
    vinculadas = materializar(
        con_jaccard(pares_v).where(f"jaccard >= {UMBRAL_VINC}")
            .join(vc.select(F.col("nombre_norm").alias("nombre_a"), F.col("clave").alias("clave_a")), "nombre_a")
            .join(vc.select(F.col("nombre_norm").alias("nombre_b"), F.col("clave").alias("clave_b")), "nombre_b")
            .where("clave_a <> clave_b"), "vinculadas")
else:
    vinculadas = spark.createDataFrame([], "nombre_b string, nombre_a string, jaccard double, clave_a string, clave_b string")

print(f"LSH listo en {time.time() - t0:,.0f} s")
mapa_sin.groupBy("metodo").count().show()
enlaces.where("jaccard < 1").orderBy("jaccard").show(10, truncate=40)   # casos frontera para validar el umbral

pendientes: 36,887 | nombres con id: 89,121 | buckets calientes: 64
LSH listo en 52 s
+------+-----+
|metodo|count|
+------+-----+
| nuevo|36776|
|exacto|  189|
|   lsh|  112|
+------+-----+

+----------------------------------------+---------------+---------------------------------------+------------------+
|                             nombre_norm|          clave|                               nombre_b|           jaccard|
+----------------------------------------+---------------+---------------------------------------+------------------+
|                    CONSORCIO ALEJANDRIA|RUC:20501519481|                  CONSORCIO ALEJANDRINO|              0.85|
|                   CONSORCIO CONSTRUCTOR|RUC:20609519194|              B Z CONSORCIO CONSTRUCTOR|              0.85|
|                    BLOOMBERG FINANCE LP|RUC:99000004533|                  BLOOMBERG FINANCE L P|              0.85|
|                CONSORCIO ASEO ASOCIADOS|RUC:20450106357|                    CONSORCIO ASOCIADOS|  

- **Por qué bandas y no `MinHashLSH` de MLlib:** MLlib solo amplifica con OR (un hash por tabla), así que nombres con
  Jaccard 0.3 colisionan con alta probabilidad y el join explota. Con b = 5 y r = 4: P(candidato | s = 0.85) ≈ 0.98 y
  P(candidato | s = 0.3) ≈ 0.04
- **Buckets calientes:** más de 500 nombres en un bucket no sirven como bloqueo (shingles comunes como "GRUPO", "CONSORCIO")
- **Shingles de 3 caracteres** (en el P1 eran 5): los nombres son cortos y k = 5 deja muy pocos shingles

#### d. Resolver tablas y dimensión de proveedores

In [12]:
t0 = time.time()
mapa_sel = mapa_sin.select("nombre_norm", F.col("clave").alias("_clave_lsh"))
mapa_b = F.broadcast(mapa_sel) if mapa_sin.count() < 500_000 else mapa_sel

def resolver(df, col_id, col_nombre):
    return (df.withColumn("_clave", clave_id(col_id))
              .withColumn("nombre_norm", normalizar_nombre(col_nombre))
              .join(mapa_b, "nombre_norm", "left")
              .withColumn("proveedor_key", F.when(F.col("_clave").isNotNull(), F.col("_clave"))
                                            .when(F.col("nombre_norm") != "", F.col("_clave_lsh")))
              .drop("_clave", "_clave_lsh", "nombre_norm"))

reg_prov = materializar(
    resolver(partes_raw.where(F.array_contains("roles", "supplier") | F.array_contains("roles", "tenderer"))
                       .select("parte_id_raw", "nombre", "region"), "parte_id_raw", "nombre")
        .where("region IS NOT NULL AND proveedor_key IS NOT NULL")
        .groupBy("proveedor_key", F.upper(F.trim("region")).alias("region")).count()
        .withColumn("rk", F.row_number().over(Window.partitionBy("proveedor_key").orderBy(F.desc("count"), "region")))
        .where("rk = 1").select("proveedor_key", "region"), "reg_prov")

dim_proveedores = materializar(
    nombres_clave.select(F.col("clave").alias("proveedor_key"), "nombre")
        .unionByName(mapa_sin.where("metodo = 'nuevo'").select(F.col("clave").alias("proveedor_key"), F.col("nombre_norm").alias("nombre")))
        .dropDuplicates(["proveedor_key"])
        .withColumn("tipo", F.split("proveedor_key", ":")[0])
        .withColumn("es_consorcio", F.col("nombre").rlike("^CONSORCIO").cast("int"))
        .join(reg_prov, "proveedor_key", "left"), "dim_proveedores")

impacto = pd.DataFrame([
    ("ids crudos distintos", menciones.select("id_raw").distinct().count()),
    ("nombres crudos distintos", menciones.select(F.upper(F.trim("nombre_raw"))).distinct().count()),
    ("claves por identificador", menciones.where("clave IS NOT NULL").select("clave").distinct().count()),
    ("claves con >1 variante de nombre", multi_variante.count()),
    ("nombres sin identificador", sin_clave.count()),
    ("  enlazados exactos", mapa_sin.where("metodo = 'exacto'").count()),
    ("  enlazados por LSH", mapa_sin.where("metodo = 'lsh'").count()),
    ("  nodos nuevos NOM:", mapa_sin.where("metodo = 'nuevo'").count()),
    ("pares posibles vinculadas (no fusionados)", vinculadas.count()),
    ("nodos proveedor finales", dim_proveedores.count())], columns=["medida", "valor"])
print(f"listo en {time.time() - t0:,.0f} s")
impacto

listo en 18 s


,medida,valor
0,ids crudos distintos,172084
1,nombres crudos distintos,127413
2,claves por identificador,88735
3,claves con >1 variante de nombre,412
4,nombres sin identificador,37077
5,enlazados exactos,189
6,enlazados por LSH,112
7,nodos nuevos NOM:,36776
8,pares posibles vinculadas (no fusionados),42
9,nodos proveedor finales,125511


- La diferencia entre "ids crudos" y "nodos finales" es lo que se habría fragmentado en el grafo sin resolución:
  ‹x› nodos menos y aristas más pesadas
- **Ética y privacidad:** las claves `PN:` contienen el DNI de personas naturales; se usan internamente para el grafo, pero
  en la web se publican hashes o agregados

---
### 3. Hechos (grano entidad–proveedor–ítem)

In [13]:
w_aw = Window.partitionBy("ocid", "award_id")
adj_r = (resolver(adj, "proveedor_id_raw", "proveedor").where("proveedor_key IS NOT NULL")
         .dropDuplicates(["ocid", "award_id", "proveedor_key"])
         .withColumn("n_proveedores_adj", F.count("*").over(w_aw)))

con_link = con_raw.select("ocid", "award_id").distinct().withColumn("tiene_contrato", F.lit(1))
fam_tender = (items_raw.where("unspsc IS NOT NULL")
              .groupBy("ocid").agg(F.array_distinct(F.collect_list(F.substring("unspsc", 1, 4))).alias("fam_tender")))
fam_award = F.array_distinct(F.transform(F.filter("unspsc_adj", lambda x: x.isNotNull()), lambda x: F.substring(x, 1, 4)))

hechos = (adj_r
    .join(proc.select("ocid", "fecha", "anio", "mes", "entidad_key", "region", "categoria", "metodo_detalle",
                      "postor_unico", "es_directa", "n_postores"), "ocid")
    .join(fam_tender, "ocid", "left")
    .join(con_link, ["ocid", "award_id"], "left").fillna({"tiene_contrato": 0})
    .withColumn("_fa", fam_award)
    .withColumn("familias", F.coalesce(F.when(F.size("_fa") > 0, F.col("_fa")),
                                       F.when(F.size("fam_tender") > 0, F.col("fam_tender")),
                                       F.array(F.lit("SIN"))))
    .withColumn("n_familias", F.size("familias"))
    .withColumn("monto_asignado", F.col("monto_adj") / F.col("n_proveedores_adj") / F.col("n_familias"))
    .select("ocid", "award_id", "fecha", "anio", "mes", "entidad_key", "proveedor_key", "region", "categoria",
            "metodo_detalle", "postor_unico", "es_directa", "n_postores", "n_proveedores_adj", "tiene_contrato",
            "monto_adj", "monto_asignado", F.explode("familias").alias("familia"))
    .withColumn("segmento", F.substring("familia", 1, 2)))

- **Criterio de arista:** la relación entidad–proveedor se define por **adjudicación válida**, no por contrato: ‹pct›% de
  adjudicaciones no tiene contrato vinculado (a_entendimiento §5.d). El contrato queda como `tiene_contrato`
- **Familia UNSPSC (4 dígitos) como ítem:** el segmento (2 dígitos) es muy grueso y 8 dígitos es muy disperso; se
  prefieren los ítems del award y, si no hay, los del tender
- **`monto_asignado` fraccional** (monto / proveedores / familias): Σ monto_asignado = monto adjudicado; si no, un consorcio
  de 3 en 4 familias contaría el monto 12 veces

#### b. Participantes (oferentes ∪ ganadores), deduplicados

In [14]:
of_r = (resolver(of_raw.join(proc.select("ocid"), "ocid"), "oferente_id_raw", "oferente")
        .where("proveedor_key IS NOT NULL").select("ocid", "proveedor_key").distinct()
        .withColumn("en_lista_oferentes", F.lit(1)))
gan = hechos.select("ocid", "proveedor_key").distinct().withColumn("gano", F.lit(1))
participantes = (of_r.join(gan, ["ocid", "proveedor_key"], "full")
                 .fillna(0, ["en_lista_oferentes", "gano"])
                 .join(proc.select("ocid", "anio", "mes"), "ocid"))
print(f"filas oferentes crudas: {of_raw.join(proc.select('ocid'), 'ocid').count():,} -> "
      f"participantes únicos: {participantes.count():,}")

filas oferentes crudas: 1,086,164 -> participantes únicos: 1,084,941


---
### 4. Escritura en `data/final` (partición anio/mes, como en el P1)

In [15]:
escribir = {"procesos": (proc, ["anio", "mes"]), "hechos": (hechos, ["anio", "mes"]),
            "participantes": (participantes, ["anio", "mes"]),
            "entidades": (dim_entidades.coalesce(1), []), "proveedores": (dim_proveedores.coalesce(1), []),
            "er_mapa_lsh": (mapa_sin.coalesce(1), []), "er_posibles_vinculadas": (vinculadas.coalesce(1), [])}
for nombre, (df, parts) in escribir.items():
    w_ = df.write.mode("overwrite")
    (w_.partitionBy(*parts) if parts else w_).parquet(str(final_dir / nombre))
print("data/final:", ", ".join(escribir))

data/final: procesos, hechos, participantes, entidades, proveedores, er_mapa_lsh, er_posibles_vinculadas


### 5. Escritura de resultados

In [16]:
tablas = {"pasos_limpieza": pasos_df, "impacto_resolucion": impacto,
          "enlaces_lsh_muestra": enlaces.orderBy("jaccard").limit(50),
          "multi_variante_muestra": multi_variante.limit(50)}
escritos = {nombre: guardar_tabla(df, nombre) for nombre, df in tablas.items()}
print(f"Escrito en 01_EDA_KDD/temp/{NOTEBOOK}/: {', '.join(escritos)}")

Escrito en 01_EDA_KDD/temp/a_transformaciones/: pasos_limpieza, impacto_resolucion, enlaces_lsh_muestra, multi_variante_muestra


#### a. Verificación

In [17]:
for nombre, n in escritos.items():
    leido = spark.read.parquet(str(tab_dir / nombre)).count()
    assert leido == n, f"{nombre}: esperaba {n}, leí {leido}"
    assert (web_dir / f"{nombre}.json").exists(), f"falta web/{nombre}.json"

figs = sorted(p.stem for p in fig_dir.glob("*.png"))
print(f"OK: {len(escritos)} tablas (parquet + json)" + (f" | {len(figs)} figuras: {', '.join(figs)}" if figs else ""))

OK: 4 tablas (parquet + json)


---
### 6. Cerrar sesión

In [18]:
spark.stop()